# Financial Analytics Engine

This notebook ingests `saas_50k_v1_with_transactions.csv`, produces structured financial facts, and writes a 112-column CSV matching the supplied `final_output.csv` schema. It contains no LLM prompts, narrative-generation logic, memory, frontend, dashboard, or chat code.

## 1. Setup and paths

In [2]:
from pathlib import Path
import json
import math
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)

INPUT_NAME = "input_data.csv"
input_candidates = [Path(INPUT_NAME), Path("outputs") / INPUT_NAME]
INPUT_CSV = next((p for p in input_candidates if p.exists()), None)
if INPUT_CSV is None:
    raise FileNotFoundError(f"Place {INPUT_NAME} beside the notebook or in ./outputs")

OUTPUT_DIR = Path("outputs") if Path("outputs").exists() else Path(".")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_CSV = OUTPUT_DIR / "finance_engine_final_output.csv"

print(f"Input:  {INPUT_CSV.resolve()}")
print(f"Output: {OUTPUT_CSV.resolve()}")

Input:  /content/input_data.csv
Output: /content/finance_engine_final_output.csv


## 2. Reusable financial analytics engine

The class implements `load_dataset()`, `compare_periods()`, `rank_variances()`, `breakdown_variance()`, `get_top_transactions()`, `get_historical_account_changes()`, and structured variance objects.

In [3]:
class FinancialAnalyticsEngine:
    """Structured financial facts only. No narrative or LLM layer."""

    OPTIONAL_DIMENSIONS = [
        "customer", "vendor", "product", "department", "region",
        "company_size", "industry", "contract_type",
    ]

    def __init__(self):
        self.summary = None
        self.transactions = None
        self.metadata = None

    @staticmethod
    def _period_from_month(month_series, start_year=2026):
        month_index = pd.to_numeric(month_series, errors="raise").astype("int64") - 1
        years = start_year + month_index // 12
        months = month_index % 12 + 1
        return years.astype(str) + "-" + months.astype(str).str.zfill(2)

    def load_dataset(self, summary_csv, transactions_csv=None):
        """Load a combined file or separate summary/transaction CSVs.

        Normalized summary columns: period, account, amount, plus dimensions.
        Normalized transaction columns: transaction_id, date, period, account,
        amount, plus dimensions that exist.
        """
        raw_summary = pd.read_csv(summary_csv)
        required_combined = {"account_id", "current_mrr"}
        missing = required_combined - set(raw_summary.columns)
        if missing:
            raise ValueError(f"Summary input is missing: {sorted(missing)}")

        if "timestamp" in raw_summary.columns:
            period = raw_summary["timestamp"].astype(str).str[:7]
        elif "period" in raw_summary.columns:
            period = raw_summary["period"].astype(str)
        elif "month" in raw_summary.columns:
            period = self._period_from_month(raw_summary["month"])
        else:
            raise ValueError("Summary input needs period, month, or timestamp")

        dimensions = [c for c in self.OPTIONAL_DIMENSIONS if c in raw_summary.columns]
        self.summary = pd.DataFrame({
            "period": period,
            "account": raw_summary["account_id"].astype(str),
            "amount": pd.to_numeric(raw_summary["current_mrr"], errors="raise"),
        })
        for dimension in dimensions:
            self.summary[dimension] = raw_summary[dimension]

        if transactions_csv is None:
            raw_transactions = raw_summary
        else:
            raw_transactions = pd.read_csv(transactions_csv)

        tx_required = {"transaction_id", "timestamp", "account_id", "amount"}
        tx_missing = tx_required - set(raw_transactions.columns)
        if tx_missing:
            raise ValueError(f"Transactions input is missing: {sorted(tx_missing)}")

        self.transactions = pd.DataFrame({
            "transaction_id": raw_transactions["transaction_id"].astype(str),
            "date": pd.to_datetime(raw_transactions["timestamp"], errors="raise"),
            "period": raw_transactions["timestamp"].astype(str).str[:7],
            "account": raw_transactions["account_id"].astype(str),
            "amount": pd.to_numeric(raw_transactions["amount"], errors="raise"),
        })
        for dimension in dimensions:
            if dimension in raw_transactions.columns:
                self.transactions[dimension] = raw_transactions[dimension]

        self.metadata = {
            "periods": sorted(self.summary["period"].dropna().unique().tolist()),
            "transaction_count": int(len(self.transactions)),
            "available_dimensions": dimensions,
        }
        return self.metadata

    def compare_periods(self, current_period, comparison_period):
        period_data = self.summary[self.summary["period"].isin([comparison_period, current_period])]
        totals = period_data.groupby(["account", "period"], observed=True)["amount"].sum().unstack(fill_value=0)
        for period in [comparison_period, current_period]:
            if period not in totals.columns:
                totals[period] = 0.0
        result = totals[[comparison_period, current_period]].reset_index()
        result.columns = ["account", "previous", "current"]
        result["change"] = result["current"] - result["previous"]
        result["change_pct"] = np.where(
            result["previous"].abs() > 1e-12,
            result["change"] / result["previous"].abs() * 100,
            np.nan,
        )
        return result.sort_values("account", kind="stable").reset_index(drop=True)

    @staticmethod
    def rank_variances(comparison, top_n=20):
        ranked = comparison.copy()
        abs_change = ranked["change"].abs()
        abs_pct = ranked["change_pct"].abs().replace([np.inf, -np.inf], np.nan).fillna(0).clip(upper=200)
        magnitude = abs_change / max(float(abs_change.max()), 1e-12)
        percentage = abs_pct / max(float(abs_pct.max()), 1e-12)
        ranked["variance_score"] = 0.7 * magnitude + 0.3 * percentage
        return ranked.sort_values(["variance_score", "change"], ascending=[False, False], kind="stable").head(top_n).reset_index(drop=True)

    def breakdown_variance(self, account, current_period, comparison_period, dimension, top_n=4):
        if dimension not in self.metadata["available_dimensions"]:
            raise ValueError(f"Dimension '{dimension}' is unavailable. Choose from {self.metadata['available_dimensions']}")
        subset = self.summary[(self.summary["account"] == str(account)) & self.summary["period"].isin([comparison_period, current_period])]
        pivot = subset.groupby([dimension, "period"], observed=True)["amount"].sum().unstack(fill_value=0)
        for period in [comparison_period, current_period]:
            if period not in pivot.columns:
                pivot[period] = 0.0
        result = (pivot[current_period] - pivot[comparison_period]).rename("change").reset_index()
        result = result.sort_values("change", key=lambda s: s.abs(), ascending=False, kind="stable")
        top = result.head(top_n).copy()
        if len(result) > top_n:
            other = pd.DataFrame([{dimension: "Other", "change": result.iloc[top_n:]["change"].sum()}])
            top = pd.concat([top, other], ignore_index=True)
        return top.reset_index(drop=True)

    def get_top_transactions(self, account, current_period, comparison_period=None, dimension=None, entity=None, top_n=10):
        periods = [current_period] if comparison_period is None else [comparison_period, current_period]
        subset = self.transactions[(self.transactions["account"] == str(account)) & self.transactions["period"].isin(periods)]
        if dimension is not None:
            if dimension not in subset.columns:
                raise ValueError(f"Dimension '{dimension}' is unavailable in transactions")
            subset = subset[subset[dimension].astype(str) == str(entity)]
        subset = subset.assign(_magnitude=subset["amount"].abs()).sort_values("_magnitude", ascending=False, kind="stable")
        return subset.drop(columns="_magnitude").head(top_n).reset_index(drop=True)

    def get_historical_account_changes(self, account, through_period=None, months=5):
        subset = self.summary[self.summary["account"] == str(account)].groupby("period", observed=True)["amount"].sum().sort_index()
        if through_period is not None:
            subset = subset.loc[subset.index <= through_period]
        result = subset.rename("amount").reset_index()
        result["change"] = result["amount"].diff()
        result["change_pct"] = result["amount"].pct_change(fill_method=None) * 100
        return result.tail(months).reset_index(drop=True)

    def build_variance_objects(self, current_period, comparison_period, dimension=None, top_n=20, drivers_per_variance=4):
        ranked = self.rank_variances(self.compare_periods(current_period, comparison_period), top_n=top_n)
        if dimension is None and self.metadata["available_dimensions"]:
            dimension = self.metadata["available_dimensions"][0]
        objects = []
        for variance_number, row in enumerate(ranked.itertuples(index=False), start=1):
            drivers = []
            if dimension is not None:
                breakdown = self.breakdown_variance(row.account, current_period, comparison_period, dimension, top_n=drivers_per_variance)
                for driver_number, driver in enumerate(breakdown.itertuples(index=False), start=1):
                    entity = getattr(driver, dimension)
                    transactions = self.get_top_transactions(row.account, current_period, comparison_period, dimension, entity)
                    drivers.append({
                        "driver_id": f"DRV_{variance_number:03d}_{driver_number:02d}",
                        "dimension": dimension,
                        "entity": entity,
                        "change": float(driver.change),
                        "transaction_ids": transactions["transaction_id"].tolist(),
                    })
            objects.append({
                "variance_id": f"VAR_{variance_number:03d}",
                "account": row.account,
                "previous": float(row.previous),
                "current": float(row.current),
                "change": float(row.change),
                "change_pct": None if pd.isna(row.change_pct) else float(row.change_pct),
                "drivers": drivers,
            })
        return objects

## 3. Build the reference-style output

The wide output is written in chunks so the full 1.15-million-row dataset does not need to exist in memory as a 112-column table.

In [4]:
FINAL_COLUMNS = [
    "account_id", "month", "company_size", "industry", "contract_type", "discount_pct", "regime_state", "active_users", "usage_growth", "feature_adoption_rate", "error_rate", "tickets_count", "ticket_growth", "payment_delay_flag", "current_mrr", "next_month_mrr", "previous_month", "previous_mrr", "previous_active_users", "previous_feature_adoption_rate", "previous_error_rate", "previous_tickets_count", "previous_discount_pct", "mrr_change", "mrr_change_pct", "absolute_mrr_change", "change_direction", "active_users_change", "active_users_change_pct", "feature_adoption_change", "error_rate_change", "tickets_change", "tickets_change_pct", "discount_change", "transaction_id", "timestamp", "transaction_type", "amount", "reason_code", "transaction_count", "transaction_ids", "transaction_timestamps", "transaction_types", "transaction_amounts", "reason_codes", "transaction_amount_total", "transaction_reconciliation_diff", "positive_transaction_amount", "negative_transaction_amount", "new_arr_amount", "expansion_amount", "contraction_amount", "churn_amount", "sla_credit_amount", "refund_amount", "usage_overage_amount", "new_arr_share", "expansion_share", "contraction_share", "churn_share", "sla_credit_share", "refund_share", "usage_overage_share", "largest_transaction_id", "largest_transaction_timestamp", "largest_transaction_type", "largest_transaction_amount", "largest_transaction_reason_code", "largest_positive_transaction_amount", "largest_negative_transaction_amount", "primary_transaction_type", "primary_reason_code", "primary_driver_amount", "primary_driver_share", "secondary_transaction_type", "secondary_reason_code", "secondary_driver_amount", "secondary_driver_share", "change_classification", "usage_signal", "adoption_signal", "reliability_signal", "support_signal", "payment_signal", "usage_growth_driver_flag", "feature_adoption_driver_flag", "error_rate_driver_flag", "ticket_growth_driver_flag", "payment_delay_driver_flag", "discount_driver_flag", "variance_score", "variance_rank", "material_variance_flag", "account_contribution_to_total_change", "account_contribution_pct", "company_size_mrr_change", "company_size_contribution_pct", "industry_mrr_change", "industry_contribution_pct", "contract_type_mrr_change", "contract_type_contribution_pct", "prior_3m_avg_mrr", "prior_3m_avg_change", "prior_3m_avg_change_pct", "prior_3m_max_change", "prior_3m_min_change", "change_vs_prior_3m_avg", "historical_change_label", "churn_flag", "churn_reason_code", "driver_summary", "evidence_summary",
]

BASE_COLUMNS = [
    "account_id", "month", "company_size", "industry", "contract_type", "discount_pct", "regime_state", "active_users", "usage_growth", "feature_adoption_rate", "error_rate", "tickets_count", "ticket_growth", "payment_delay_flag", "current_mrr", "next_month_mrr", "transaction_id", "timestamp", "transaction_type", "amount", "reason_code",
]

def _safe_divide(numerator, denominator):
    denominator = pd.to_numeric(denominator, errors="coerce")
    numerator = pd.to_numeric(numerator, errors="coerce")
    return numerator.div(denominator.where(denominator.abs() > 1e-12))

def _build_global_metrics(input_csv):
    usecols = ["account_id", "month", "company_size", "industry", "contract_type", "current_mrr", "next_month_mrr"]
    metrics = pd.read_csv(input_csv, usecols=usecols)
    metrics["mrr_change"] = metrics["next_month_mrr"] - metrics["current_mrr"]
    metrics["mrr_change_pct"] = _safe_divide(metrics["mrr_change"], metrics["current_mrr"].abs())
    absolute = metrics["mrr_change"].abs()
    max_absolute = absolute.groupby(metrics["month"], observed=True).transform("max").replace(0, np.nan)
    metrics["variance_score"] = (0.7 * absolute.div(max_absolute).fillna(0) + 0.3 * metrics["mrr_change_pct"].abs().clip(upper=2).div(2).fillna(0))
    metrics["variance_rank"] = metrics.groupby("month", observed=True)["variance_score"].rank(method="first", ascending=False).astype("int64")
    metrics["month_total_change"] = metrics.groupby("month", observed=True)["mrr_change"].transform("sum")
    metrics["month_count"] = metrics.groupby("month", observed=True)["mrr_change"].transform("size")
    for dimension in ["company_size", "industry", "contract_type"]:
        metrics[f"{dimension}_mrr_change"] = metrics.groupby(["month", dimension], observed=True)["mrr_change"].transform("sum")
    return metrics[["variance_score", "variance_rank", "month_total_change", "month_count", "company_size_mrr_change", "industry_mrr_change", "contract_type_mrr_change"]]

def _enrich_complete_accounts(block, global_metrics):
    block = block.copy()
    positions = block.pop("_source_row").to_numpy(dtype="int64")
    metrics = global_metrics.iloc[positions].reset_index(drop=True)
    block = block.reset_index(drop=True)
    group = block.groupby("account_id", sort=False, observed=True)

    previous_map = {
        "previous_month": "month",
        "previous_mrr": "current_mrr",
        "previous_active_users": "active_users",
        "previous_feature_adoption_rate": "feature_adoption_rate",
        "previous_error_rate": "error_rate",
        "previous_tickets_count": "tickets_count",
        "previous_discount_pct": "discount_pct",
    }
    for output, source in previous_map.items():
        block[output] = group[source].shift(1)

    block["mrr_change"] = block["next_month_mrr"] - block["current_mrr"]
    block["mrr_change_pct"] = _safe_divide(block["mrr_change"], block["current_mrr"].abs())
    block["absolute_mrr_change"] = block["mrr_change"].abs()
    block["change_direction"] = np.select([block["mrr_change"] > 1e-12, block["mrr_change"] < -1e-12], ["increase", "decrease"], default="flat")
    block["active_users_change"] = block["active_users"] - block["previous_active_users"]
    block["active_users_change_pct"] = _safe_divide(block["active_users_change"], block["previous_active_users"].abs())
    block["feature_adoption_change"] = block["feature_adoption_rate"] - block["previous_feature_adoption_rate"]
    block["error_rate_change"] = block["error_rate"] - block["previous_error_rate"]
    block["tickets_change"] = block["tickets_count"] - block["previous_tickets_count"]
    block["tickets_change_pct"] = _safe_divide(block["tickets_change"], block["previous_tickets_count"].abs())
    block["discount_change"] = block["discount_pct"] - block["previous_discount_pct"]

    amount = pd.to_numeric(block["amount"], errors="raise")
    tx_type = block["transaction_type"].astype(str)
    block["transaction_count"] = 1
    block["transaction_ids"] = '["' + block["transaction_id"].astype(str) + '"]'
    block["transaction_timestamps"] = '["' + block["timestamp"].astype(str) + '"]'
    block["transaction_types"] = '["' + tx_type + '"]'
    block["transaction_amounts"] = "[" + amount.astype(str) + "]"
    block["reason_codes"] = '["' + block["reason_code"].astype(str) + '"]'
    block["transaction_amount_total"] = amount
    block["transaction_reconciliation_diff"] = block["mrr_change"] - amount
    block["positive_transaction_amount"] = amount.clip(lower=0)
    block["negative_transaction_amount"] = amount.clip(upper=0)

    type_specs = [
        ("New_ARR", "new_arr"), ("Expansion", "expansion"), ("Contraction", "contraction"),
        ("Churn", "churn"), ("SLA_Credit", "sla_credit"), ("Refund", "refund"),
        ("Usage_Overage", "usage_overage"),
    ]
    denominator = amount.abs().replace(0, np.nan)
    for transaction_type, prefix in type_specs:
        value = np.where(tx_type.eq(transaction_type), amount, 0.0)
        block[f"{prefix}_amount"] = value
        block[f"{prefix}_share"] = pd.Series(np.abs(value), index=block.index).div(denominator).fillna(0)

    block["largest_transaction_id"] = block["transaction_id"]
    block["largest_transaction_timestamp"] = block["timestamp"]
    block["largest_transaction_type"] = tx_type
    block["largest_transaction_amount"] = amount
    block["largest_transaction_reason_code"] = block["reason_code"]
    block["largest_positive_transaction_amount"] = amount.clip(lower=0)
    block["largest_negative_transaction_amount"] = amount.clip(upper=0)
    block["primary_transaction_type"] = tx_type
    block["primary_reason_code"] = block["reason_code"]
    block["primary_driver_amount"] = amount
    block["primary_driver_share"] = 1.0
    block["secondary_transaction_type"] = "None"
    block["secondary_reason_code"] = ""
    block["secondary_driver_amount"] = 0.0
    block["secondary_driver_share"] = 0.0
    block["change_classification"] = tx_type

    block["usage_signal"] = np.select([block["usage_growth"] > 0.05, block["usage_growth"] < -0.05], ["positive", "negative"], default="stable")
    block["adoption_signal"] = np.select([block["feature_adoption_change"] > 0.02, block["feature_adoption_change"] < -0.02], ["positive", "negative"], default="stable")
    block["reliability_signal"] = np.select([block["error_rate_change"] < -0.02, block["error_rate_change"] > 0.02], ["positive", "negative"], default="stable")
    block["support_signal"] = np.select([block["tickets_change"] < 0, block["tickets_change"] > 0], ["positive", "negative"], default="stable")
    block["payment_signal"] = np.where(block["payment_delay_flag"].astype(bool), "negative", "stable")
    block["usage_growth_driver_flag"] = block["usage_growth"].abs() >= 0.05
    block["feature_adoption_driver_flag"] = block["feature_adoption_change"].abs() >= 0.02
    block["error_rate_driver_flag"] = block["error_rate_change"].abs() >= 0.02
    block["ticket_growth_driver_flag"] = block["ticket_growth"].abs() >= 0.25
    block["payment_delay_driver_flag"] = block["payment_delay_flag"].astype(bool)
    block["discount_driver_flag"] = block["discount_change"].abs() >= 0.01

    block["variance_score"] = metrics["variance_score"].to_numpy()
    block["variance_rank"] = metrics["variance_rank"].to_numpy()
    material_cutoff = np.maximum(10, np.ceil(metrics["month_count"].to_numpy() * 0.10))
    block["material_variance_flag"] = (block["variance_rank"].to_numpy() <= material_cutoff) & (block["absolute_mrr_change"] > 0)
    block["account_contribution_to_total_change"] = block["mrr_change"]
    block["account_contribution_pct"] = _safe_divide(block["mrr_change"], metrics["month_total_change"])
    for dimension in ["company_size", "industry", "contract_type"]:
        total_column = f"{dimension}_mrr_change"
        block[total_column] = metrics[total_column].to_numpy()
        block[f"{dimension}_contribution_pct"] = _safe_divide(block[total_column], metrics["month_total_change"])

    previous_mrr = group["current_mrr"].shift(1)
    previous_change = group["mrr_change"].shift(1)
    previous_change_pct = group["mrr_change_pct"].shift(1)
    by_account = block["account_id"]
    block["prior_3m_avg_mrr"] = previous_mrr.groupby(by_account, observed=True).rolling(3, min_periods=1).mean().reset_index(level=0, drop=True)
    block["prior_3m_avg_change"] = previous_change.groupby(by_account, observed=True).rolling(3, min_periods=1).mean().reset_index(level=0, drop=True)
    block["prior_3m_avg_change_pct"] = previous_change_pct.groupby(by_account, observed=True).rolling(3, min_periods=1).mean().reset_index(level=0, drop=True)
    block["prior_3m_max_change"] = previous_change.groupby(by_account, observed=True).rolling(3, min_periods=1).max().reset_index(level=0, drop=True)
    block["prior_3m_min_change"] = previous_change.groupby(by_account, observed=True).rolling(3, min_periods=1).min().reset_index(level=0, drop=True)
    prior_count = previous_change.groupby(by_account, observed=True).rolling(3, min_periods=1).count().reset_index(level=0, drop=True)
    block["change_vs_prior_3m_avg"] = block["mrr_change"] - block["prior_3m_avg_change"]
    unusual = block["mrr_change"].abs() > (2 * block["prior_3m_avg_change"].abs().clip(lower=0.01))
    block["historical_change_label"] = np.select(
        [prior_count < 3, unusual & (block["mrr_change"] > block["prior_3m_avg_change"]), unusual & (block["mrr_change"] < block["prior_3m_avg_change"])],
        ["insufficient_history", "unusual_increase", "unusual_decrease"],
        default="normal",
    )
    block["churn_flag"] = tx_type.eq("Churn") | (block["next_month_mrr"] <= 0)
    block["churn_reason_code"] = np.where(block["churn_flag"], block["reason_code"], "")
    signed_amount = [f"{value:+.4f}" for value in amount.to_numpy()]
    block["driver_summary"] = tx_type + " " + pd.Series(signed_amount, index=block.index)
    block["evidence_summary"] = "1 transactions; largest " + block["transaction_id"].astype(str) + " " + pd.Series(signed_amount, index=block.index) + " (" + block["reason_code"].astype(str) + ")"

    missing = [column for column in FINAL_COLUMNS if column not in block.columns]
    if missing:
        raise AssertionError(f"Output columns were not built: {missing}")
    return block[FINAL_COLUMNS]

def build_final_output(input_csv=INPUT_CSV, output_csv=OUTPUT_CSV, chunk_size=100_000):
    """Create the 112-column reference-style output in bounded-memory chunks."""
    global_metrics = _build_global_metrics(input_csv)
    required = set(BASE_COLUMNS)
    header = pd.read_csv(input_csv, nrows=0).columns.tolist()
    missing = sorted(required - set(header))
    if missing:
        raise ValueError(f"Input is missing required columns: {missing}")

    if Path(output_csv).exists():
        Path(output_csv).unlink()
    pending = None
    source_offset = 0
    written = 0
    first_write = True
    for chunk in pd.read_csv(input_csv, chunksize=chunk_size):
        chunk["_source_row"] = np.arange(source_offset, source_offset + len(chunk), dtype="int64")
        source_offset += len(chunk)
        if pending is not None:
            chunk = pd.concat([pending, chunk], ignore_index=True)
        last_account = chunk["account_id"].iloc[-1]
        complete = chunk[chunk["account_id"] != last_account]
        pending = chunk[chunk["account_id"] == last_account].copy()
        if not complete.empty:
            enriched = _enrich_complete_accounts(complete, global_metrics)
            enriched.to_csv(output_csv, mode="w" if first_write else "a", header=first_write, index=False, float_format="%.10g")
            written += len(enriched)
            first_write = False
            print(f"Wrote {written:,} rows")
    if pending is not None and not pending.empty:
        enriched = _enrich_complete_accounts(pending, global_metrics)
        enriched.to_csv(output_csv, mode="w" if first_write else "a", header=first_write, index=False, float_format="%.10g")
        written += len(enriched)

    if written != len(global_metrics):
        raise AssertionError(f"Expected {len(global_metrics):,} rows, wrote {written:,}")
    output_header = pd.read_csv(output_csv, nrows=0).columns.tolist()
    if output_header != FINAL_COLUMNS:
        raise AssertionError("Final output header does not match the reference schema")
    return {"output_csv": str(Path(output_csv).resolve()), "rows": written, "columns": len(FINAL_COLUMNS)}

build_result = build_final_output()
build_result

Wrote 99,981 rows
Wrote 199,985 rows
Wrote 299,989 rows
Wrote 399,993 rows
Wrote 499,997 rows
Wrote 599,978 rows
Wrote 699,982 rows
Wrote 799,986 rows
Wrote 899,990 rows
Wrote 999,994 rows
Wrote 1,099,998 rows
Wrote 1,149,977 rows


{'output_csv': '/content/finance_engine_final_output.csv',
 'rows': 1150000,
 'columns': 112}

## 4. Structured engine examples

In [5]:
# Load the generated output into the reusable engine and show structured examples.
engine = FinancialAnalyticsEngine()
metadata = engine.load_dataset(INPUT_CSV)
periods = metadata["periods"]
comparison_period, current_period = periods[-2], periods[-1]
comparison = engine.compare_periods(current_period, comparison_period)
top_variances = engine.rank_variances(comparison, top_n=10)
historical_example = engine.get_historical_account_changes(top_variances.iloc[0]["account"], through_period=current_period)

print(json.dumps(metadata, indent=2))
display(top_variances)
display(historical_example)
structured_variances = engine.build_variance_objects(current_period, comparison_period, dimension="industry", top_n=3)
print(json.dumps(structured_variances[:1], indent=2))

{
  "periods": [
    "2026-01",
    "2026-02",
    "2026-03",
    "2026-04",
    "2026-05",
    "2026-06",
    "2026-07",
    "2026-08",
    "2026-09",
    "2026-10",
    "2026-11",
    "2026-12",
    "2027-01",
    "2027-02",
    "2027-03",
    "2027-04",
    "2027-05",
    "2027-06",
    "2027-07",
    "2027-08",
    "2027-09",
    "2027-10",
    "2027-11"
  ],
  "transaction_count": 1150000,
  "available_dimensions": [
    "company_size",
    "industry",
    "contract_type"
  ]
}


,account,previous,current,change,change_pct,variance_score
0,8593,101241.608433,171608.971153,70367.362720,69.504390,0.804257
1,10643,39879.180632,84980.146114,45100.965482,113.094012,0.618296
2,3836,57311.673296,103383.598330,46071.925034,80.388379,0.578897
3,22122,21116.297608,54585.371809,33469.074201,158.498781,0.570692
4,20820,131948.513333,82757.215961,-49191.297372,-37.280676,0.545266
5,25767,57449.175879,90171.185897,32722.010018,56.958189,0.410949
6,15452,23939.493443,48707.140456,24767.647013,103.459361,0.401572
7,49455,47884.628151,74067.063784,26182.435633,54.678164,0.342475
8,40305,2925.357063,8475.959767,5550.602705,189.741033,0.339828
9,45801,1083.376688,4400.227286,3316.850598,306.158572,0.332995


,period,amount,change,change_pct
0,2027-07,44028.326208,-570.071973,-1.278234
1,2027-08,62796.643402,18768.317194,42.627824
2,2027-09,73802.717541,11006.074139,17.526533
3,2027-10,101241.608433,27438.890891,37.178700
4,2027-11,171608.971153,70367.362720,69.504390


[
  {
    "variance_id": "VAR_001",
    "account": "8593",
    "previous": 101241.6084325358,
    "current": 171608.97115296312,
    "change": 70367.36272042732,
    "change_pct": 69.50439034887312,
    "drivers": [
      {
        "driver_id": "DRV_001_01",
        "dimension": "industry",
        "entity": "Logistics",
        "change": 70367.36272042732,
        "transaction_ids": [
          "CR-000197662",
          "INV-000197661"
        ]
      }
    ]
  }
]


## 5. Output validation

In [6]:
# Final validation against the requested output contract.
output_header = pd.read_csv(OUTPUT_CSV, nrows=0).columns.tolist()
assert output_header == FINAL_COLUMNS
assert len(output_header) == 112
row_count = sum(1 for _ in open(OUTPUT_CSV, "rb")) - 1
assert row_count == 1_150_000
sample = pd.read_csv(OUTPUT_CSV, nrows=5)
assert sample["transaction_id"].notna().all()
assert sample["transaction_type"].isin(["New_ARR", "Expansion", "Contraction", "Churn", "SLA_Credit", "Refund", "Usage_Overage"]).all()
print({"rows": row_count, "columns": len(output_header), "output": str(OUTPUT_CSV.resolve())})
display(sample)

{'rows': 1150000, 'columns': 112, 'output': '/content/finance_engine_final_output.csv'}


,account_id,month,company_size,industry,contract_type,discount_pct,regime_state,active_users,usage_growth,feature_adoption_rate,error_rate,tickets_count,ticket_growth,payment_delay_flag,current_mrr,next_month_mrr,previous_month,previous_mrr,previous_active_users,previous_feature_adoption_rate,previous_error_rate,previous_tickets_count,previous_discount_pct,mrr_change,mrr_change_pct,absolute_mrr_change,change_direction,active_users_change,active_users_change_pct,feature_adoption_change,error_rate_change,tickets_change,tickets_change_pct,discount_change,transaction_id,timestamp,transaction_type,amount,reason_code,transaction_count,transaction_ids,transaction_timestamps,transaction_types,transaction_amounts,reason_codes,transaction_amount_total,transaction_reconciliation_diff,positive_transaction_amount,negative_transaction_amount,new_arr_amount,expansion_amount,contraction_amount,churn_amount,sla_credit_amount,refund_amount,usage_overage_amount,new_arr_share,expansion_share,contraction_share,churn_share,sla_credit_share,refund_share,usage_overage_share,largest_transaction_id,largest_transaction_timestamp,largest_transaction_type,largest_transaction_amount,largest_transaction_reason_code,largest_positive_transaction_amount,largest_negative_transaction_amount,primary_transaction_type,primary_reason_code,primary_driver_amount,primary_driver_share,secondary_transaction_type,secondary_reason_code,secondary_driver_amount,secondary_driver_share,change_classification,usage_signal,adoption_signal,reliability_signal,support_signal,payment_signal,usage_growth_driver_flag,feature_adoption_driver_flag,error_rate_driver_flag,ticket_growth_driver_flag,payment_delay_driver_flag,discount_driver_flag,variance_score,variance_rank,material_variance_flag,account_contribution_to_total_change,account_contribution_pct,company_size_mrr_change,company_size_contribution_pct,industry_mrr_change,industry_contribution_pct,contract_type_mrr_change,contract_type_contribution_pct,prior_3m_avg_mrr,prior_3m_avg_change,prior_3m_avg_change_pct,prior_3m_max_change,prior_3m_min_change,change_vs_prior_3m_avg,historical_change_label,churn_flag,churn_reason_code,driver_summary,evidence_summary
0,0,1,SMB,Energy,Monthly,0.054167,stable,4.765884,0.017574,0.002766,0.102689,0,0.0,0,112.703497,50.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-62.703497,-0.556358,62.703497,decrease,NaN,NaN,NaN,NaN,NaN,NaN,NaN,CR-000000001,2026-01-14T11:22:58,Contraction,-62.703497,Contract_Downgrade,1,"[""CR-000000001""]","[""2026-01-14T11:22:58""]","[""Contraction""]",[-62.7034973284],"[""Contract_Downgrade""]",-62.703497,2.935252e-11,0.000000,-62.703497,0,0.000000,-62.703497,0,0.000000,0,0,0,0,1,0,0,0,0,CR-000000001,2026-01-14T11:22:58,Contraction,-62.703497,Contract_Downgrade,0.000000,-62.703497,Contraction,Contract_Downgrade,-62.703497,1,NaN,NaN,0,0,Contraction,stable,stable,stable,stable,stable,False,False,False,False,False,False,0.084878,13682,False,-62.703497,3.392921e-06,-1.242946e+06,0.067257,-1.715075e+06,0.092804,-8713910.247,0.471514,NaN,NaN,NaN,NaN,NaN,NaN,insufficient_history,False,NaN,Contraction -62.7035,1 transactions; largest CR-000000001 -62.7035 ...
1,0,2,SMB,Energy,Monthly,0.054167,stable,11.706351,0.027244,0.000000,0.102174,2,inf,1,50.000000,51.903945,1.0,112.703497,4.765884,0.002766,0.102689,0.0,0.054167,1.903945,0.038079,1.903945,increase,6.940467,1.456281,-0.002766,-0.000515,2.0,NaN,0.0,INV-000000002,2026-02-15T16:29:11,Expansion,1.903945,Seat_Add,1,"[""INV-000000002""]","[""2026-02-15T16:29:11""]","[""Expansion""]",[1.9039450193],"[""Seat_Add""]",1.903945,2.608758e-11,1.903945,0.000000,0,1.903945,0.000000,0,0.000000,0,0,0,1,0,0,0,0,0,INV-000000002,2026-02-15T16:29:11,Expansion,1.903945,Seat_Add,1.903945,0.000000,Expansion,Seat_Add,1.903945,1,NaN,NaN,0,0,Expansion,stable,stable,stable,negative,negative,False,False,False,True,True,False,0.005733,40498,False,1.903945,-1.161215e-07,-1.565495e+06,0.095479,-1.750803e+06,0.106781,-8911671.018,0.543523,112.703497,-62.703497,-0.556358,-6